# PPI-KG
+ GE & PPI-KG overlapping proteins: 8085
+ GE proteins: 8085 + 12480 = 20565
+ PPI-KG proteins: 8085 + 543 = 8628

### BUT the PPI-KG I have can't match the same protein counts  

# Sample Scoring
+ ecdf;
+ healthy control as reference distribution;
+ 1%, 1.5%, 2.5%, 5%, 10%, 20%

# Generate Network
+ my PNG.generate() generate the same results as CLEP's network generator

In [1]:
import os
import sys

try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))

from data_processing.pyg_graph_generator import *

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
output_dir = "../CLEP_replication/networks/PPI_KGs_my"
dataset = 'adni'
process_method = 'DiseaseKG'
scoring_method = 'ecdf1'

kg_disease_path = "../datasets/base_kgs/ppi_hc.pkl"
kg_health_path = "../data/KG/healthy_aging_reversed_remove_noncausal.pkl"
exp_path = "../data/ADNI/cleaned_gene_expression_data.csv"
scoring_path = "../data/ADNI/old_target/ecdf_1/sample_scoring_ecdf.csv"

In [10]:
os.makedirs(output_dir, exist_ok=True)
save_network = os.path.join(output_dir, f"G_{dataset}_{process_method}_{scoring_method}.pkl")
save_summary = os.path.join(output_dir, f"Summary_{dataset}_{process_method}_{scoring_method}.csv")

# 1. Load expression df, smaple scoring df, KG
exp_df = pd.read_csv(exp_path, index_col=0)
if exp_df.shape[0] > exp_df.shape[1]:
    exp_df = exp_df.transpose()
data = pd.read_csv(scoring_path, index_col=0)
kg_disease = load_graph(kg_disease_path)
kg_control = load_graph(kg_health_path)

# clean exp_df before K-NN
# drop genes with no variation
exp_df = exp_df.loc[:, exp_df.std() > 0]
# Using median is usually safer for gene expression
exp_df = exp_df.fillna(exp_df.median())
# normalize safely
min_val = exp_df.min()
max_val = exp_df.max()
exp_norm = (exp_df - min_val) / (max_val - min_val + 1e-9)
# final fill-na
exp_norm = exp_norm.fillna(0)

Loaded graph from ../datasets/base_kgs/ppi_hc.pkl: 17042 nodes, 382526 edges
Loaded graph from ../data/KG/healthy_aging_reversed_remove_noncausal.pkl: 4161 nodes, 13775 edges


# RotatE 
+ best hyperparameters are in the config dict

In [ ]:
# best hyperparameters for RotatE

config = {
    "model": "RotatE",
    "training_loop": "sLCWA",  # Stochastic Local Closed World Assumption
    
    # Model-specific parameters
    "model_kwargs": {
        "embedding_dim": 128,  
    },
    
    # Loss function and its specific parameters
    "loss": "NSSALoss",  # Negative Sampling Self-Adversarial Loss
    "loss_kwargs": {
        "margin": 23.92,
        "adversarial_temperature": 0.93,
    },
    
    # Negative Sampler configuration (used by sLCWA)
    "negative_sampler_kwargs": {
        "num_negatives_per_positive": 22,
    },
    
    # Training runtime execution parameters
    "training_kwargs": {
        "num_epochs": 200,
        "batch_size": 1024,
    }
}

In [3]:
from CLEP_replication.embedding.kge import *

In [ ]:
graph = load_graph("../CLEP_replication/networks/PPI_KGs_my/G_adni_DiseaseKG_ecdf_1.5.pkl")
design_path = "../data/ADNI/design_with_real_target.tsv"
design = pd.read_csv(design_path, index_col=0, sep='\t')


Loaded graph from ../CLEP_replication/networks/PPI_KGs_my/G_adni_DiseaseKG_ecdf_1.5.pkl: 17786 nodes, 714714 edges


,Old_Target,Target
FileName,,
116_S_1249,Control,AD
037_S_4410,Control,Control
006_S_4153,Disease,AD
116_S_1232,Control,Control
099_S_4205,Disease,MCI
...,...,...
009_S_2381,Disease,AD
053_S_4557,Disease,MCI
073_S_4300,Disease,MCI


# Classification: Logistic Regression